# QC Processing Notebook
Development version that expects access to QC files via local filesystem

Breaks out the individual steps from the wrapper workflow.qc_process_and_model() so you can troubleshoot it step by step

In [1]:
import os
import sys
from pathlib import Path
from earthscope_sfg_workflows.workflows.workflow_handler import WorkflowHandler
from earthscope_sfg_workflows.data_mgmt.model import GARPOSLayout
from earthscope_sfg_workflows.pipelines.config import ( SVPConfig,
    QCPipelineConfig, QCPinConfig, RinexConfig, PrideConfig, PositionUpdateConfig, KinConfig
)
import logging
from earthscope_sfg_workflows.logging.loggers import set_all_logger_levels
logging.basicConfig(level=logging.INFO)
set_all_logger_levels(logging.INFO)


import pandas as pd
import matplotlib.pyplot as plt
import warnings
logging.getLogger("gnss_product_management").setLevel(logging.ERROR)
warnings.filterwarnings("ignore", category=FutureWarning, module="earthscope_sfg_tools.tiledb_integration.arrays")

#required for local garpos to work, not needed in GEOLAB (Should confirm)
os.environ["GARPOS_PATH"] = "/Users/mikegottlieb/GIT/earthscope-sfg-workflows/.pixi/garpos/bin"

# GARPOS's internal multiprocessing.Pool uses macOS's default "spawn" start method.
# CPython's multiprocessing/spawn.py snapshots the PARENT's sys.path at Pool() time
# and forces the CHILD to use that exact snapshot (sys.path = data['sys_path'] in
# spawn.prepare()) - it does NOT let the child compute its own sys.path from
# PYTHONPATH at startup. So os.environ["PYTHONPATH"] alone does nothing here; only
# a direct sys.path mutation in this (already-running) kernel is captured by that
# snapshot and propagated to spawned workers.
GARPOS_BIN = "/Users/mikegottlieb/GIT/earthscope-sfg-workflows/.pixi/garpos/bin"
if GARPOS_BIN not in sys.path:
    sys.path.insert(0, GARPOS_BIN)

# VS Code's Jupyter kernel launches .pixi/envs/geolab/bin/python directly, bypassing
# pixi's [tool.pixi.activation] env vars (same reason GARPOS_PATH has to be set above).
# Prepend the pixi-managed PRIDE-PPPAR build so it's found before any stale ~/.PRIDE_PPPAR_BIN.
os.environ["PATH"] = "/Users/mikegottlieb/GIT/earthscope-sfg-workflows/.pixi/.PRIDE_PPPAR_BIN" + os.pathsep + os.environ["PATH"]

## Set these parameters

In [ ]:
# Main working directory for all data operations
main_dir = Path("/Users/mikegottlieb/data/sfg")

# Network, station, and campaign identifiers
NETWORK = "cascadia-gorda"
STATION = "TEST"
CAMPAIGN = "2026_B_1126"

# Directory containing raw QC .pin files to ingest
#raw_qc_data_dir = Path("/Users/mikegottlieb/data/PSN011153_all_data/NTH1_2025/20250812")

# Optional overrides to force rerunning individual steps even if they have been previously done
override_steps = {
    "generate_svp": True,
    "process_qc_pin": True,
    "build_rinex": True,
    "run_pride": True,
    "process_kinematic": True,
    "refine_shotdata": True
}

# =========================================================================
# Initialize Workflow
# =========================================================================

# Create the workflow handler
workflow = WorkflowHandler(directory=main_dir)

# Set the processing context (network/station/campaign)
workflow.set_network_station_campaign(
    network_id=NETWORK,
    station_id=STATION,
    campaign_id=CAMPAIGN,
)

pipeline = workflow._session.pipeline           # ProcessingService for the active scope


cfg  = QCPipelineConfig(qcpin_config=QCPinConfig(override=override_steps["process_qc_pin"]),
                        rinex_config=RinexConfig(override=override_steps["build_rinex"], time_interval=24),
                        pride_config=PrideConfig(override=override_steps["run_pride"]),
                        kinematic_config=KinConfig(override=override_steps["process_kinematic"]),
                        position_update_config=PositionUpdateConfig(override=override_steps["refine_shotdata"]),
                        svp_config=SVPConfig(override=override_steps["generate_svp"]))

qc = workflow._session.pipeline.get_qc()

/Users/mikegottlieb/GIT/earthscope-sfg-workflows/src/earthscope_sfg_workflows/workflows/workspace.py:88: UserWarning: Environment variable S3_SYNC_BUCKET is not set.
  Environment.load_working_environment()
Active context: cascadia-gorda / NCC1 / 2025_A_1126


In [ ]:

# =========================================================================
# Ingest QC Data (if not already done)
# =========================================================================

# Ingest raw QC files from local directory
# This step adds .pin files to the asset catalog
workflow.ingest_qc()    


In [ ]:
pipeline.run_qc("process_qcpin",   config=cfg)  # pins -> shotdata + writes qc_gnss_obs.tdb

In [ ]:
pipeline.run_qc("build_rinex", config=cfg)  # qc_gnss_obs.tdb -> daily RINEX (the tdb2rnx step)

In [ ]:
pipeline.run_qc("run_pride", config=cfg)  # RINEX -> KIN + residuals (PRIDE-PPP)

In [ ]:
pipeline.run_qc("process_kinematic", config=cfg)  # KIN -> kinematic-position DataFrame 


In [ ]:
pipeline.run_qc("refine_shotdata", config=cfg)  # merge positions into final shotdata

# Now do QC checks

In [ ]:
# 

In [ ]:
# plot PRIDE results
 
from pride_ppp.factories.output import read_kin_data, get_wrms_from_res, plot_kin_results_wrms

intermediate_dir = main_dir / NETWORK / STATION / CAMPAIGN / "intermediate"

for kin_path in sorted(intermediate_dir.glob("kin_*.kin")):
    res_path = kin_path.parent / f"{kin_path.stem.replace('kin_', 'res_')}.res"

    kin_df = read_kin_data(kin_path)
    kin_df.index = kin_df.index.tz_localize("UTC")  # match get_wrms_from_res's tz-aware index

    if res_path.exists():
        wrms_df = get_wrms_from_res(res_path).set_index("date")
        kin_df = pd.merge_asof(
            kin_df.sort_index(), wrms_df.sort_index(),
            left_index=True, right_index=True,
            direction="nearest", tolerance=pd.Timedelta(seconds=0.01),
        )
    else:
        kin_df["wrms"] = float("nan")

    plot_kin_results_wrms(kin_df, title=str(kin_path))


In [ ]:
# check kin completeness vs rinex files
from earthscope_sfg_workflows.pipelines.plotting import plot_kin_position_data
from earthscope_sfg_workflows.data_mgmt.model import AssetKind

rinex_entries = qc.catalog.assets_to_process(
    kind=AssetKind.RINEX4,  # your files are RINEX 4.02 per the header
    override=True,
    network=NETWORK, station=STATION, campaign=CAMPAIGN,
)
plot_kin_position_data(qc.qcKinPositionTDB, rinex_entries=rinex_entries)